# 05 — Flink Aggregation Pipeline (SDK + raw jobs API)

avg(temperature), 1-min tumbling window, burst of 3, results assert, DELETE + re-GET 404 proof.
No SDK Flink model exists — job lifecycle uses the SDK's authenticated `request()` (same auth).

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" python-dotenv

In [ ]:
import os, random, time
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()
BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
assert BASE_URL and USERNAME and PASSWORD, "SKIP: set CENOTOO_BASE_URL/USERNAME/PASSWORD"
PIPE_TEMPS = [20.0, 22.0, 24.0]
EXPECTED_AVG = sum(PIPE_TEMPS) / len(PIPE_TEMPS)
JOB_BODY = {"attribute": "temperature", "metric": "avg", "window_type": "tumbling", "window_size": 1, "unit": "minute"}
def unique_name(p="nb"):
    return f"{p}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}"
def _jobs_path(pid, cid, suffix=""):
    return f"/api/v1/projects/{pid}/collections/{cid}/jobs{suffix}"
def wait_for(fn, pred, timeout=300, interval=5, desc="c"):
    import time as _t
    end = _t.monotonic() + timeout
    while _t.monotonic() < end:
        v = fn()
        try:
            if pred(v):
                return v
        except Exception:
            pass
        _t.sleep(interval)
    raise TimeoutError(desc)

In [ ]:
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType
from nostradamus_ioto_sdk.exceptions import APIError, ResourceNotFoundError
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
project = admin.projects.create(name=unique_name("pipe"), description="nb05 pipeline", tags=["notebook", "05"])
rk = admin.project_keys.create(project.project_id, KeyType.READ)
wk = admin.project_keys.create(project.project_id, KeyType.WRITE)
collection = admin.collections.create(project_id=project.project_id, name=unique_name("pipecoll"),
    description="nb05", tags=["notebook", "05"],
    collection_schema={"key": "PIPE_001", "timestamp": "2026-01-01T00:00:00Z", "temperature": 20.0, "humidity": 50.0})
print("setup OK")

In [ ]:
try:
    job = admin.request("POST", _jobs_path(project.project_id, collection.collection_id), json=JOB_BODY).json()
except APIError as e:
    if getattr(e, "status_code", None) == 503:
        print("SKIP: Flink SQL Gateway unavailable")
        raise SystemExit(0)
    raise
job_id = job["id"]
assert job["status"] in ("RUNNING", "PENDING"), job
print("job:", job_id, job["status"])
# Single back-to-back burst, ONE shared key (job GROUPs BY key -> record_count == 3).
# Minute-boundary straddle risk is negligible (single produce call, milliseconds).
probe = unique_name("pipeburst")
base = datetime.now(timezone.utc).replace(microsecond=0)
admin.data.send(project_id=project.project_id, collection_id=collection.collection_id,
    data=[{"key": probe, "timestamp": base.isoformat().replace("+00:00", "Z"),
             "temperature": t, "humidity": 50.0} for t in PIPE_TEMPS])
print("burst sent")

In [ ]:
res = wait_for(lambda: admin.request("GET", _jobs_path(project.project_id, collection.collection_id, f"/{job_id}/results")).json(),
    lambda j: any((r.get("record_count") or 0) >= 3 for r in j.get("results", [])), desc="aggregation results")
vals = [r["value"] for r in res["results"] if (r.get("record_count") or 0) >= 3]
assert any(abs(v - EXPECTED_AVG) < 1e-6 for v in vals), (vals, EXPECTED_AVG)
print(f"aggregation OK: avg={EXPECTED_AVG}")

In [ ]:
admin.request("DELETE", _jobs_path(project.project_id, collection.collection_id, f"/{job_id}"))
try:
    admin.request("GET", _jobs_path(project.project_id, collection.collection_id, f"/{job_id}"))
    raise AssertionError("expected 404 after DELETE")
except ResourceNotFoundError:
    print("deletion proof OK: re-GET -> 404")

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    admin.collections.delete(project.project_id, collection.collection_id)
with contextlib.suppress(Exception):
    admin.projects.delete(project.project_id)
admin.close()
print("cleanup done")